### data\fred_fetcher.py 
This code is a template for downloading data from the Federal Reserve Economic Data (FRED) API. It includes a function to fetch data for a given series ID and date range, and a main block to execute the function with example parameters. The code uses the `requests` library to make HTTP requests to the FRED API and handles the response by checking for errors and printing the data.

In [ ]:
import requests
import pandas as pd
from dotenv import load_dotenv
import os
load_dotenv()
API_KEY = os.getenv("FRED_API_KEY_LB")
FRED_LIST= 'fred_daily_series_list.csv'

def get_daily_series(api_key):
    base_url = "https://api.stlouisfed.org/fred/tags/series"
    params = {
        "api_key": api_key,
        "tag_names": "daily",
        "file_type": "json",
        "limit": 1000,
        "offset": 0
    }

    all_series = []

    while True:
        try:
            response = requests.get(base_url, params=params)
            response.raise_for_status()
            data = response.json()

            if "seriess" in data:
                series_chunk = data["seriess"]
                all_series.extend(series_chunk)

                if len(series_chunk) < params["limit"]:
                    break
                params["offset"] += params["limit"]
            else:
                break
        except requests.exceptions.RequestException as e:
            print(f"Error fetching data: {e}")
            break
    return all_series



daily_series = get_daily_series(API_KEY)
df = pd.DataFrame(daily_series)
print(f"\nNumber of FRED series with daily data: {len(df)}\n")
df = df[['id']]
df.to_csv(FRED_LIST, index=False)

import os
from io import StringIO

START_DATE = '2000-01-01'
END_DATE = '2024-12-31'
DATA_FOLDER = 'testrun'

def fetch_data(series_id):
    request = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
    request += f"&cosd={START_DATE}"
    request += f"&coed={END_DATE}"
    try:
        response = requests.get(request)
        response.raise_for_status()
        df = pd.read_csv(StringIO(response.text), parse_dates=True)
        df.rename(
            columns={
                df.columns[0]: 'ds',
                df.columns[1]: 'value',
            },
            inplace=True,
        )
        return series_id, df
    except requests.RequestException as e:
        print(f"Error fetching data for {series_id}: {e}")
        return series_id, None

df = pd.read_csv(FRED_LIST)
os.makedirs(DATA_FOLDER, exist_ok=True)
series_ids = df['id'].tolist()

for series_id in series_ids:
    series_id, data = fetch_data(series_id)
    if data is not None:
        filename = os.path.join(DATA_FOLDER, f"{series_id}.csv")
        data.to_csv(filename, index=False)


def count_csv_files(folder_path):
    csv_count = 0
    for filename in os.listdir(folder_path):
        if filename.endswith('.csv'):
            csv_count += 1
    return csv_count

num_csv_files = count_csv_files(DATA_FOLDER)
print(f"\nNumber of CSV files in the {DATA_FOLDER} folder: {num_csv_files}\n")


file_path = os.path.join(DATA_FOLDER, 'AAA10Y.csv')
df = pd.read_csv(file_path)

print("\nLast 10 rows of AAA10Y:")
print("=" * 30)
print(df.tail(10))
print("=" * 30)

# import pandas_market_calendars as mcal
# from typing import Optional


# def obtain_market_dates(start_date: str, end_date: str, market : Optional[str] = "NYSE") -> pd.DataFrame:
#     nyse = mcal.get_calendar(market)
#     market_open_dates = nyse.schedule(
#         start_date=start_date,
#         end_date=end_date,
#     )
#     return market_open_dates


# market_dates = obtain_market_dates(START_DATE,END_DATE)

# def replace_empty_data(df : pd.DataFrame) -> pd.DataFrame:
#     mask = df.isin(["", ".", None])
#     rows_to_remove = mask.any(axis=1)
#     return df.loc[~rows_to_remove]


# from typing import Union, Tuple
# import logging
# MAX_MISSING_DATA = 0.02

# def handle_missing_data(
#         data: pd.DataFrame,
#         market_open_dates : pd.DataFrame,
#         data_series : str
# ) -> Tuple[Union[None,pd.DataFrame], Union[pd.DataFrame, None]]:
#     modified_data = data.copy()
#     market_open_dates["count"] = 0
#     date_counts = data['ds'].value_counts()

#     market_open_dates["count"] = market_open_dates.index.map(
#         date_counts
#     ).fillna(0)

#     missing_dates = market_open_dates.loc[
#         market_open_dates["count"] < 1
#     ]

#     if not missing_dates.empty:
#         max_count = (
#             len(market_open_dates)
#             * MAX_MISSING_DATA
#         )

#         if len(missing_dates) > max_count:
#             logging.warning(
#                 f"For the asset {data_series} there are "
#                 f"{len(missing_dates)} data points missing, which is greater than the maximum threshold of "
#                 f"{MAX_MISSING_DATA * 100}%"
#             )
#             return pd.DataFrame(), None
#         else:
#             for date, row in missing_dates.iterrows():
#                 modified_data = insert_missing_date(
#                     modified_data, date, 'ds'
#                 )
#     return modified_data, missing_dates


# def insert_missing_date(
#         data: pd.DataFrame,
#         date: str,
#         date_column: str
# ) -> pd.DataFrame:
#     date = pd.to_datetime(date)
#     if date not in data[date_column].values:
#         prev_date = (
#             data[data[date_column] < date].iloc[-1]
#             if not data[data[date_column] < date].empty
#             else data.iloc[0]
#         )
#         new_row = prev_date.copy()
#         new_row[date_column] = date
#         data = (
#             pd.concat([data, new_row.to_frame().T], ignore_index=True)
#             .sort_values(by=date_column)
#             .reset_index(drop=True)
#         )
#     return data


# import glob
# processed_dataframes = []
# market_dates_only = market_dates.index.date

# for csv_file in glob.glob(os.path.join(DATA_FOLDER, "*.csv")):

#     df = pd.read_csv(csv_file)
#     df['ds'] = pd.to_datetime(df['ds'])
#     df_correct_dates = df[df['ds'].dt.date.isin(market_dates_only)]
#     df_cleaned = replace_empty_data(df_correct_dates)
#     processed_df, missing_dates = handle_missing_data(df_cleaned,market_dates,os.path.basename(csv_file).split('.')[0])
#     if not processed_df.empty:
#         processed_df['ds'] = pd.to_datetime(processed_df['ds'])
#         if not missing_dates.empty:
#             for missing_date in missing_dates.index:
#                 missing_date = pd.to_datetime(missing_date)
#                 if missing_date in processed_df['ds'].values:
#                     continue

#                 previous_day_data = processed_df[processed_df['ds'] < missing_date].tail(1)

#                 if previous_day_data.empty:
#                     new_row = pd.DataFrame({'ds': [missing_date], 'value': [0]})
#                 else:
#                     new_row = previous_day_data.copy()
#                     new_row['ds'] = missing_date

#                 processed_df = pd.concat([processed_df, new_row]).sort_values('ds').reset_index(drop=True)
#         if 'SP500.csv' in csv_file:
#             model_data = processed_df.rename(columns={'value': 'price'}).reset_index(drop=True)
#         processed_dataframes.append(processed_df.reset_index(drop=True))

# print(f"\nNumber of data series remaining after cleanup: {len(processed_dataframes)}\n")


# from sklearn.preprocessing import StandardScaler
# from sklearn.decomposition import PCA
# EXPLAINED_VARIANCE = .9
# MIN_VARIANCE = 1e-10

# combined_df = pd.concat([df.set_index('ds') for df in processed_dataframes], axis=1)
# combined_df.columns = [f'value_{i}' for i in range(len(processed_dataframes))]

# X = combined_df.values
# scaler = StandardScaler()
# X_scaled = scaler.fit_transform(X)


# pca = PCA(n_components=EXPLAINED_VARIANCE, svd_solver='full')
# X_pca = pca.fit_transform(X_scaled)

# X_pca = X_pca[:, pca.explained_variance_ > MIN_VARIANCE]


# pca_df = pd.DataFrame(
#     X_pca,
#     columns=[f'PC{i+1}' for i in range(X_pca.shape[1])]
# )

# print(f"\nOriginal number of features: {combined_df.shape[1]}")
# print(f"Number of components after PCA: {pca_df.shape[1]}\n")

# model_data = model_data.join(pca_df)


# from neuralforecast.models import TFT
# from neuralforecast import NeuralForecast

# TRAIN_SIZE = .90
# model_data['unique_id'] = 'SPY'
# model_data['price'] = model_data['price'].astype(float)
# model_data['y'] = model_data['price'].pct_change()
# model_data = model_data.iloc[1:]
# hist_exog_list = [col for col in model_data.columns if col.startswith('PC')]

# train_size = int(len(model_data) * TRAIN_SIZE)
# train_data = model_data[:train_size]
# test_data = model_data[train_size:]

# model = TFT(
#     h=1,
#     input_size=24,
#     hist_exog_list=hist_exog_list,
#     scaler_type='robust',
#     max_steps=20

# )

# nf = NeuralForecast(
#     models=[model],
#     freq='D'
# )

# nf.fit(df=model_data)


# y_hat_test_ret = pd.DataFrame()
# current_train_data = train_data.copy()

# y_hat_ret = nf.predict(current_train_data)
# y_hat_test_ret = pd.concat([y_hat_test_ret, y_hat_ret.iloc[[-1]]])

# for i in range(len(test_data) - 1):
#     combined_data = pd.concat([current_train_data, test_data.iloc[[i]]])
#     y_hat_ret = nf.predict(combined_data)
#     y_hat_test_ret = pd.concat([y_hat_test_ret, y_hat_ret.iloc[[-1]]])
#     current_train_data = combined_data

# predicted_returns = y_hat_test_ret['TFT'].values

# predicted_prices_ret = []
# for i, ret in enumerate(predicted_returns):
#     if i == 0:
#         last_true_price = train_data['price'].iloc[-1]
#     else:
#         last_true_price = test_data['price'].iloc[i-1]
#     predicted_prices_ret.append(last_true_price * (1 + ret))


# import matplotlib.pyplot as plt
# true_values = test_data['price']

# plt.figure(figsize=(12, 6))
# plt.plot(train_data['ds'], train_data['price'], label='Training Data', color='blue')
# plt.plot(test_data['ds'], true_values, label='True Prices', color='green')
# plt.plot(test_data['ds'], predicted_prices_ret, label='Predicted Prices', color='red')
# plt.legend()
# plt.title('Basic SPY Stepwise Forecast using TFT')
# plt.xlabel('Date')
# plt.ylabel('SPY Price')
# plt.savefig('spy_forecast_chart.png', dpi=300, bbox_inches='tight')
# plt.close()


### data\fred_grand_pipeline.py

In [ ]:
#!/usr/bin/env python3
"""
FRED Grand Pipeline — Final Version

1. Downloads ALL ~11,000 daily FRED series (if not already downloaded)
2. Deletes series that DON'T cover 2000-01-01 to 2024-12-31
3. Filters to interpretable/important series
4. Builds a grand daily table
5. Engineers macro/regime features

Only CSV output. No parquet.

Usage:
    python data/fred_grand_pipeline.py                  # Use existing files
    python data/fred_grand_pipeline.py --download       # Download + process
    python data/fred_grand_pipeline.py --force-refresh  # Redownload everything
"""

from __future__ import annotations

import argparse
import json
import os
import shutil
import time
from io import StringIO
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv
from tqdm import tqdm

load_dotenv()
API_KEY = os.getenv("FRED_API_KEY_LB", "")
DEBUG_MODE = bool(int(os.getenv("DEBUG_MODE", 1)))

if not API_KEY:
    raise SystemExit("FRED_API_KEY_LB not found in .env file")

# ============================================================
# PATHS
# ============================================================

# DATA_PATH = Path.home() / "remoteRepo" / "data" / "FRED_data"
DATA_PATH1 = Path("data/FRED_data")
RAW_DIR = DATA_PATH1 / "raw"
OUTPUT_DIR = DATA_PATH1 / "outputs"
FRED_LIST_FILE = DATA_PATH1 / "fred_daily_series_list.csv"

# ============================================================
# DATE RANGE (STRICT)
# ============================================================

TARGET_START = pd.Timestamp("2000-01-01")
TARGET_END = pd.Timestamp("2024-12-31")

# ============================================================
# SERIES SELECTION — Only keep interpretable, useful series
# ============================================================

# These MUST be available. If not in the daily tag, we download them separately.
REQUIRED_SERIES = [
    "VIXCLS",      # VIX (may not be tagged "daily")
    "T10Y2Y",      # 10Y-2Y spread
    "T10Y3M",      # 10Y-3M spread
    "USRECD",      # Recession indicator
    "TEDRATE",     # TED spread
    "DTWEXBGS",    # Trade-weighted dollar
]

KEEP_PATTERNS = [
    # === Treasury Yields ===
    "DGS1", "DGS2", "DGS3", "DGS5", "DGS7", "DGS10", "DGS20", "DGS30",
    "DGS1MO", "DGS3MO", "DGS6MO",
    
    # === Treasury Bills ===
    "DTB1YR", "DTB3", "DTB6",
    
    # === Yield Spreads (pre-computed) ===
    "T10Y2Y", "T10Y3M", "T10YFF", "T1YFF", "T3MFF", "T5YFF", "T6MFF",
    
    # === Corporate Bonds ===
    "AAA10Y", "AAAFF", "BAA10Y", "BAAFF", "DAAA", "DBAA",
    
    # === Stock Indices ===
    "NASDAQCOM", "NASDAQ100", "NIKKEI225",
    
    # === Exchange Rates ===
    "DEXUSEU", "DEXJPUS", "DEXUSUK", "DEXCAUS", "DEXCHUS",
    "DEXMXUS", "DEXKOUS", "DEXINUS",
    
    # === Commodities ===
    "DCOILWTICO", "DCOILBRENTEU", "DDFUELLA",
    "DGASNYH", "DGASUSGULF", "DHOILNYH",
    "DPROPANEMBTX", "DHHNGSP",
    
    # === Volatility ===
    "VIXCLS", "VXDCLS",
    
    # === Central Bank Rates ===
    "DFF", "DPRIME", "ECBDFR", "ECBMLFR", "ECBMRRFR",
    
    # === Policy Uncertainty ===
    "USEPUINDXD", "WLEMUINDXD",
    
    # === Recession ===
    "USRECD", "USRECDM", "USRECDP",
    
    # === Weekly/Monthly ===
    "ICSA", "MORTGAGE30US",
    
    # === Commercial Paper ===
    "CPFF", "DCPF1M", "DCPF2M", "DCPF3M", "DCPN2M", "DCPN30", "DCPN3M",
    
    # === Japan Rates ===
    "JPINTDDMEJPY", "JPINTDEXR", "JPINTDUSDJPY", "JPINTDUSDRP",
    
    # === TED Spread ===
    "TEDRATE",
    
    # === Trade-Weighted Dollar ===
    "DTWEXBGS",
    
    # === Other useful ===
    "KCPRU", "DLTIIT", "IUDSOIA", "INFECTDISEMVTRACKD",
]

# Series to ALWAYS exclude (even if they match KEEP_PATTERNS partially)
EXCLUDE_PATTERNS = [
    "BAML", "NASDAQNQ", "NASDAQHX", "NASDAQCX", "NASDAQSX",
    "NASDAQIX", "NASDAQB", "THREEFF", "THREEFY", "RIFSPP",
    "RPAGYD", "RPMBSD", "RPON", "RPTM", "RPTSYD", "RPTTLD",
    "DEXBZUS", "DEXDNUS", "DEXHKUS", "DEXMAUS", "DEXNOUS",
    "DEXSDUS", "DEXSFUS", "DEXSIUS", "DEXSLUS", "DEXSZUS",
    "DEXTAUS", "DEXTHUS", "DEXUSAL", "DEXUSNZ", "DEXVZUS",
    "DJFUELUSGULF",
]


def should_keep(series_id: str) -> bool:
    """Check if a series should be kept based on patterns."""
    for pattern in EXCLUDE_PATTERNS:
        if series_id.startswith(pattern):
            return False
    for pattern in KEEP_PATTERNS:
        if series_id == pattern or series_id.startswith(pattern):
            return True
    return False


# ============================================================
# HELPERS
# ============================================================

def now_ts() -> float:
    return time.time()


def fmt_elapsed(seconds: float) -> str:
    if seconds < 60:
        return f"{seconds:.1f}s"
    minutes = seconds / 60.0
    if minutes < 60:
        return f"{minutes:.1f}m"
    hours = minutes / 60.0
    return f"{hours:.2f}h"


# ============================================================
# STEP 0: DOWNLOAD ALL DAILY FRED SERIES
# ============================================================

def get_daily_series_list(api_key: str) -> list[str]:
    """Fetch list of ALL daily FRED series IDs from the API."""
    base_url = "https://api.stlouisfed.org/fred/tags/series"
    params = {
        "api_key": api_key,
        "tag_names": "daily",
        "file_type": "json",
        "limit": 1000,
        "offset": 0,
    }
    
    all_series = []
    
    while True:
        try:
            resp = requests.get(base_url, params=params, timeout=30)
            resp.raise_for_status()
            data = resp.json()
            
            if "seriess" in data:
                chunk = data["seriess"]
                all_series.extend(chunk)
                if len(chunk) < params["limit"]:
                    break
                params["offset"] += params["limit"]
            else:
                break
        except Exception as e:
            print(f"Error fetching series list: {e}")
            break
    
    series_ids = sorted(set(s["id"] for s in all_series))
    return series_ids


def download_single_series(series_id: str) -> Optional[pd.DataFrame]:
    """Download a single FRED series."""
    url = (
        f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
        f"&cosd=2000-01-01&coed=2024-12-31"
    )
    try:
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        df = pd.read_csv(StringIO(resp.text), parse_dates=True)
        if len(df.columns) >= 2:
            df = df.iloc[:, :2]
            df.columns = ["observation_date", series_id]
            df["observation_date"] = pd.to_datetime(df["observation_date"])
            return df
        return None
    except Exception as e:
        if DEBUG_MODE:
            print(f"[DEBUG]: Download failed for {series_id}: {e}")
        return None


def download_all_series(force_refresh: bool = False) -> None:
    """Download all daily FRED series + required series."""
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    
    existing = list(RAW_DIR.glob("*.csv"))
    print(f"Existing files in {RAW_DIR}: {len(existing):,}")
    
    # Get list of all daily-tagged series
    series_ids = get_daily_series_list(API_KEY)
    print(f"Total daily FRED series available: {len(series_ids):,}")
    
    # Add required series that might not be tagged "daily"
    all_to_download = set(series_ids)
    for sid in REQUIRED_SERIES:
        if sid not in all_to_download:
            all_to_download.add(sid)
            if DEBUG_MODE:
                print(f"[DEBUG]: Adding required series not in daily tag: {sid}")
    
    all_to_download = sorted(all_to_download)
    
    # Save the list
    pd.DataFrame({"id": all_to_download}).to_csv(FRED_LIST_FILE, index=False)
    print(f"Saved series list to: {FRED_LIST_FILE}")
    
    if not force_refresh and len(existing) > 5000:
        # Only download missing required series
        missing_required = [s for s in REQUIRED_SERIES if not (RAW_DIR / f"{s}.csv").exists()]
        if missing_required:
            print(f"Downloading {len(missing_required)} missing required series: {missing_required}")
            for sid in tqdm(missing_required, desc="Downloading required"):
                df = download_single_series(sid)
                if df is not None and len(df) > 0:
                    df.to_csv(RAW_DIR / f"{sid}.csv", index=False)
                time.sleep(0.1)
        else:
            print("All required series present. Skipping download.")
        return
    
    # Download all missing series
    new_downloads = 0
    for sid in tqdm(all_to_download, desc="Downloading FRED series"):
        csv_path = RAW_DIR / f"{sid}.csv"
        if csv_path.exists() and not force_refresh:
            continue
        
        df = download_single_series(sid)
        if df is not None and len(df) > 0:
            df.to_csv(csv_path, index=False)
            new_downloads += 1
        time.sleep(0.02)
    
    print(f"New downloads: {new_downloads:,}")
    final_count = len(list(RAW_DIR.glob("*.csv")))
    print(f"Total files in {RAW_DIR}: {final_count:,}")


# ============================================================
# STEP 1: DELETE SERIES THAT DON'T COVER 2000-2024
# ============================================================

def delete_incomplete_series() -> tuple[int, int]:
    """
    Scan ALL CSV files, delete those that don't cover 2000-01-01 to 2024-12-31.
    """
    print(f"\n{'=' * 60}")
    print("SCANNING & DELETING SERIES OUTSIDE 2000-2024 RANGE")
    print(f"{'=' * 60}")
    
    csv_files = sorted(RAW_DIR.glob("*.csv"))
    kept = 0
    deleted = 0
    trash_dir = DATA_PATH / "deleted_incomplete"
    trash_dir.mkdir(exist_ok=True)
    
    for csv_path in tqdm(csv_files, desc="Checking date ranges"):
        try:
            # Read just the date column
            df = pd.read_csv(csv_path)
            date_col = None
            for col in df.columns:
                if "date" in col.lower():
                    date_col = col
                    break
            if date_col is None:
                shutil.move(str(csv_path), str(trash_dir / csv_path.name))
                deleted += 1
                continue
            
            df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
            df = df.dropna(subset=[date_col])
            
            if df.empty:
                shutil.move(str(csv_path), str(trash_dir / csv_path.name))
                deleted += 1
                continue
            
            actual_start = df[date_col].min()
            actual_end = df[date_col].max()
            
            starts_ok = actual_start <= TARGET_START + pd.Timedelta(days=15)
            ends_ok = actual_end >= TARGET_END
            
            if starts_ok and ends_ok:
                kept += 1
            else:
                shutil.move(str(csv_path), str(trash_dir / csv_path.name))
                deleted += 1
        
        except Exception:
            shutil.move(str(csv_path), str(trash_dir / csv_path.name))
            deleted += 1
    
    print(f"\nKept (2000-2024): {kept}")
    print(f"Deleted (moved to {trash_dir}): {deleted}")
    return kept, deleted


# ============================================================
# STEP 2: FILTER TO USEFUL SERIES
# ============================================================

def filter_useful_series() -> list[str]:
    """Filter remaining series to only the useful/interpretable ones."""
    csv_files = sorted(RAW_DIR.glob("*.csv"))
    
    kept_series = []
    excluded_series = []
    
    for csv_path in csv_files:
        sid = csv_path.stem
        if should_keep(sid):
            kept_series.append(sid)
        else:
            excluded_series.append(sid)
    
    print(f"\n{'=' * 60}")
    print("FILTERING TO INTERPRETABLE SERIES")
    print(f"{'=' * 60}")
    print(f"Remaining after date check: {len(csv_files)}")
    print(f"Kept (useful): {len(kept_series)}")
    print(f"Excluded (redundant/granular): {len(excluded_series)}")
    
    if DEBUG_MODE:
        print(f"\n[DEBUG] Kept: {', '.join(sorted(kept_series))}")
    
    return kept_series


# ============================================================
# STEP 3: BUILD GRAND TABLE
# ============================================================

def build_grand_table(series_ids: list[str]) -> pd.DataFrame:
    """Load kept series and merge into a single daily table."""
    daily_cal = pd.date_range(start="2000-01-01", end="2024-12-31", freq="D")
    grand = pd.DataFrame({"observation_date": daily_cal})  # <-- FIX: use observation_date
    
    loaded = 0
    failed = 0
    missing_files = []
    
    for sid in tqdm(series_ids, desc="Building grand table"):
        csv_path = RAW_DIR / f"{sid}.csv"
        
        if not csv_path.exists():
            missing_files.append(sid)
            failed += 1
            continue
        
        try:
            df = pd.read_csv(csv_path)
            
            # Standardize: the date column is always "observation_date"
            date_col = "observation_date"
            
            if date_col not in df.columns:
                failed += 1
                continue
            
            # The other column is the value
            val_cols = [c for c in df.columns if c != date_col]
            if not val_cols:
                failed += 1
                continue
            
            # Keep only the two columns we need
            df = df[[date_col, val_cols[0]]].copy()
            df.columns = ["observation_date", sid]  # Rename to standard
            
            df["observation_date"] = pd.to_datetime(df["observation_date"], errors="coerce")
            df = df.dropna(subset=["observation_date"])
            
            # Merge
            grand = grand.merge(df, on="observation_date", how="left")
            loaded += 1
        
        except Exception as e:
            if DEBUG_MODE:
                print(f"[DEBUG]: Error loading {sid}: {e}")
            failed += 1
    
    grand = grand.set_index("observation_date")
    grand.index.name = "date"
    
    print(f"\nLoaded: {loaded} | Failed: {failed}")
    if missing_files:
        print(f"Missing files: {missing_files}")
    print(f"Grand table shape: {grand.shape}")
    
    return grand

# ============================================================
# STEP 4: SMART FORWARD-FILL
# ============================================================

def smart_fill(df: pd.DataFrame) -> pd.DataFrame:
    """Forward-fill with limits based on series frequency."""
    result = df.copy()
    
    # Determine fill limit by counting non-NaN values
    for col in df.columns:
        n_valid = df[col].count()
        total_days = len(df)
        ratio = n_valid / total_days
        
        if ratio < 0.05:   # ~300 values over 25 years = monthly
            result[col] = df[col].ffill(limit=90)
        elif ratio < 0.15:  # ~1300 values = weekly
            result[col] = df[col].ffill(limit=7)
        # Daily (>15% coverage): no fill needed
    
    return result


# ============================================================
# STEP 5: FEATURE ENGINEERING
# ============================================================

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Create derived macro/regime features."""
    feats = pd.DataFrame(index=df.index)
    
    # Yield Spreads
    if all(c in df.columns for c in ["DGS10", "DGS2"]):
        feats["yield_spread_10y2y"] = df["DGS10"] - df["DGS2"]
    if all(c in df.columns for c in ["DGS10", "DGS3MO"]):
        feats["yield_spread_10y3m"] = df["DGS10"] - df["DGS3MO"]
    if all(c in df.columns for c in ["DGS2", "DFF"]):
        feats["rate_spread_2y_fed"] = df["DGS2"] - df["DFF"]
    if all(c in df.columns for c in ["BAA10Y", "AAA10Y"]):
        feats["credit_spread_baa_aaa"] = df["BAA10Y"] - df["AAA10Y"]
    if all(c in df.columns for c in ["BAA10Y", "DGS10"]):
        feats["credit_spread_corp_treasury"] = df["BAA10Y"] - df["DGS10"]
    
    # VIX Z-Scores
    if "VIXCLS" in df.columns:
        for w in [20, 60, 252]:
            roll_mean = df["VIXCLS"].rolling(w).mean()
            roll_std = df["VIXCLS"].rolling(w).std().replace(0, np.nan)
            feats[f"vix_z_{w}d"] = (df["VIXCLS"] - roll_mean) / roll_std
    
    # Dollar/Yen/Oil Changes
    if "DEXUSEU" in df.columns:
        feats["dollar_chg_20d"] = df["DEXUSEU"].pct_change(20) * 100
    if "DEXJPUS" in df.columns:
        feats["yen_chg_20d"] = df["DEXJPUS"].pct_change(20) * 100
    if "DCOILWTICO" in df.columns:
        feats["oil_chg_20d"] = df["DCOILWTICO"].pct_change(20) * 100
    
    # Stock Returns
    for idx in ["NASDAQCOM", "NASDAQ100", "NIKKEI225"]:
        if idx in df.columns:
            feats[f"{idx}_ret_20d"] = df[idx].pct_change(20) * 100
    
    # Regime Flags
    if "yield_spread_10y2y" in feats.columns:
        feats["regime_yield_inverted"] = (feats["yield_spread_10y2y"] < 0).astype(int)
    if "VIXCLS" in df.columns:
        feats["regime_high_vix"] = (df["VIXCLS"] > 25).astype(int)
        feats["regime_extreme_vix"] = (df["VIXCLS"] > 35).astype(int)
    if "USRECD" in df.columns:
        feats["regime_recession"] = df["USRECD"].fillna(0).astype(int)
    
    # TED Spread Z-Score
    if "TEDRATE" in df.columns:
        roll_mean = df["TEDRATE"].rolling(60).mean()
        roll_std = df["TEDRATE"].rolling(60).std().replace(0, np.nan)
        feats["ted_z_60d"] = (df["TEDRATE"] - roll_mean) / roll_std
    
    # Policy Uncertainty
    if "USEPUINDXD" in df.columns:
        feats["epu_chg_20d"] = df["USEPUINDXD"].pct_change(20) * 100
    
    if DEBUG_MODE:
        print(f"[DEBUG]: Engineered {len(feats.columns)} derived features")
    
    result = pd.concat([df, feats], axis=1)
    return result


# ============================================================
# STEP 6: FINAL CLEAN
# ============================================================

def final_clean(df: pd.DataFrame) -> pd.DataFrame:
    """Drop columns with >60% missing, forward-fill, truncate start."""
    if df.empty:
        print("ERROR: Empty DataFrame. Cannot clean.")
        return df
    
    missing_pct = df.isna().mean()
    cols_to_drop = missing_pct[missing_pct > 0.60].index.tolist()
    
    if DEBUG_MODE and cols_to_drop:
        print(f"[DEBUG]: Dropping {len(cols_to_drop)} columns (>60% missing)")
    
    df = df.drop(columns=cols_to_drop, errors='ignore')
    
    if df.empty:
        print("ERROR: All columns dropped. Check your data.")
        return df
    
    df = df.ffill()
    
    completeness = df.notna().mean(axis=1)
    valid_mask = completeness > 0.3
    
    if valid_mask.any():
        valid_start = completeness[valid_mask].index[0]
        df = df[df.index >= valid_start]
    
    df = df.dropna()
    
    if DEBUG_MODE:
        print(f"[DEBUG]: Final shape: {df.shape}")
    
    return df


# ============================================================
# MAIN
# ============================================================

def main():
    parser = argparse.ArgumentParser(description="FRED Grand Pipeline")
    parser.add_argument("--download", action="store_true", help="Download missing series")
    parser.add_argument("--force-refresh", action="store_true", help="Redownload all series")
    args = parser.parse_args()
    
    overall_start = now_ts()
    
    print("=" * 60)
    print("FRED GRAND PIPELINE — Complete")
    print(f"Date range: {TARGET_START.strftime('%Y-%m-%d')} → {TARGET_END.strftime('%Y-%m-%d')}")
    print("=" * 60)
    
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    # ── Step 0: Download ──
#    if args.download or args.force_refresh:
#        print("\n[Step 0/6] Downloading FRED daily series...")
#        t0 = now_ts()
#        download_all_series(force_refresh=args.force_refresh)
#        print(f"  Done in {fmt_elapsed(now_ts() - t0)}")
#    else:
#        existing = len(list(RAW_DIR.glob("*.csv")))
#        print(f"\n[Step 0/6] Skipping download. {existing:,} files already exist.")
#        print("  Use --download to download missing series.")
    
    # # ── Step 1: Delete incomplete ──
    # print("\n[Step 1/6] Deleting series outside 2000-2024 range...")
    # t0 = now_ts()
    # kept, deleted = delete_incomplete_series()
    # print(f"  Kept: {kept} | Deleted: {deleted}")
    # print(f"  Done in {fmt_elapsed(now_ts() - t0)}")
    
    # ── Step 2: Filter useful ──
    print("\n[Step 2/6] Filtering to interpretable series...")
    t0 = now_ts()
    useful_series = filter_useful_series()
    print(f"  Done in {fmt_elapsed(now_ts() - t0)}")
    
    if not useful_series:
        print("ERROR: No useful series found. Check your KEEP_PATTERNS.")
        return
    
    # ── Step 3: Build grand table ──
    print(f"\n[Step 3/6] Building grand table from {len(useful_series)} series...")
    t0 = now_ts()
    grand = build_grand_table(useful_series)
    print(f"  Done in {fmt_elapsed(now_ts() - t0)}")
    
    if grand.empty or grand.shape[1] == 0:
        print("ERROR: Grand table is empty. Cannot proceed.")
        return
    
    # Save raw grand table
    grand.to_csv(OUTPUT_DIR / "grand_table_raw.csv")
    
    # ── Step 4: Smart fill ──
    print("\n[Step 4/6] Smart forward-filling...")
    t0 = now_ts()
    grand_filled = smart_fill(grand)
    print(f"  Done in {fmt_elapsed(now_ts() - t0)}")
    
    # ── Step 5: Feature engineering ──
    print("\n[Step 5/6] Engineering derived features...")
    t0 = now_ts()
    all_features = engineer_features(grand_filled)
    all_features.to_csv(OUTPUT_DIR / "macro_all_features.csv")
    print(f"  Total columns: {len(all_features.columns)}")
    print(f"  Done in {fmt_elapsed(now_ts() - t0)}")
    
    # ── Step 6: Final clean ──
    print("\n[Step 6/6] Final cleaning...")
    t0 = now_ts()
    clean = final_clean(all_features)
    
    # if clean.empty:
    #     print("ERROR: Final DataFrame is empty after cleaning.")
    #     return
    
    # clean.to_csv(OUTPUT_DIR / "daily_macro_features_clean.csv")
    # print(f"  Final shape: {clean.shape}")
    # print(f"  Date range: {clean.index[0].strftime('%Y-%m-%d')} → {clean.index[-1].strftime('%Y-%m-%d')}")
    # print(f"  Done in {fmt_elapsed(now_ts() - t0)}")
    
    # ── Summary ──
    total_elapsed = now_ts() - overall_start
    
    print(f"\n{'=' * 60}")
    print(f"PIPELINE COMPLETE — {fmt_elapsed(total_elapsed)}")
    print(f"Output: {OUTPUT_DIR / 'daily_macro_features_clean.csv'}")
    print(f"Series kept: {len(useful_series)}")
    print(f"Final features: {clean.shape[1]}")
    print(f"Final rows: {clean.shape[0]}")
    print(f"{'=' * 60}")
    
    # Save metadata
    metadata = {
        "date_range": [str(clean.index[0]), str(clean.index[-1])],
        "n_series_kept": len(useful_series),
        "n_features_final": int(clean.shape[1]),
        "n_rows": int(clean.shape[0]),
        "series_list": sorted(useful_series),
        "feature_names": list(clean.columns),
    }
    with open(OUTPUT_DIR / "pipeline_metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)
    print(f"Metadata: {OUTPUT_DIR / 'pipeline_metadata.json'}")
    
    # Sample
    print("\nSample (first 5 rows, first 10 columns):")
    print(clean.iloc[:5, :10])


if __name__ == "__main__":
    main()



### data\fred_clean_pipeline.py

In [ ]:
#!/usr/bin/env python3
"""
FRED Data Cleaning Pipeline

Takes the grand table and:
1. Filters to NYSE trading days only (removes weekends/holidays)
2. Drops columns with excessive missing data
3. Generates summary statistics
4. Outputs a clean, trading-day-only macro features file

Usage:
    python data/fred_clean_pipeline.py
"""

from __future__ import annotations

import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd

DEBUG_MODE = bool(int(os.getenv("DEBUG_MODE", 1)))

# ============================================================
# PATHS
# ============================================================

DATA_PATH = Path("data/FRED_data")
GRAND_TABLE = DATA_PATH / "outputs" / "grand_table_raw.csv"
MARKET_DATES = Path("data/market_dates_NYSE.csv")
OUTPUT_DIR = DATA_PATH / "outputs"

# ============================================================
# HELPERS
# ============================================================

def now_ts() -> float:
    return time.time()


def fmt_elapsed(seconds: float) -> str:
    if seconds < 60:
        return f"{seconds:.1f}s"
    minutes = seconds / 60.0
    if minutes < 60:
        return f"{minutes:.1f}m"
    hours = minutes / 60.0
    return f"{hours:.2f}h"


# ============================================================
# STEP 1: LOAD GRAND TABLE + MARKET DATES
# ============================================================

def load_data() -> tuple[pd.DataFrame, pd.DatetimeIndex]:
    """Load the grand table and NYSE trading calendar."""
    
    # Load grand table
    print(f"Loading grand table: {GRAND_TABLE}")
    df = pd.read_csv(GRAND_TABLE, index_col=0, parse_dates=True)
    print(f"  Shape: {df.shape}")
    print(f"  Date range: {df.index[0]} → {df.index[-1]}")
    
    # Load NYSE market dates
    print(f"\nLoading market dates: {MARKET_DATES}")
    market_dates_df = pd.read_csv(MARKET_DATES)
    
    # Find the date column (first column)
    date_col = market_dates_df.columns[0]
    print(f"  Date column: '{date_col}'")
    
    # Parse dates - just take the date part, ignore time
    market_dates = pd.to_datetime(market_dates_df[date_col]).dt.normalize()
    
    # Filter to 2000-2024 range
    market_dates = market_dates[
        (market_dates >= "2000-01-01") & (market_dates <= "2024-12-31")
    ]
    
    print(f"  Trading days (2000-2024): {len(market_dates):,}")
    
    return df, pd.DatetimeIndex(market_dates)

# ============================================================
# STEP 2: FILTER TO TRADING DAYS ONLY
# ============================================================

def filter_trading_days(df: pd.DataFrame, market_dates: pd.DatetimeIndex) -> pd.DataFrame:
    """Keep only rows that fall on NYSE trading days."""
    
    # Convert market dates to a set of date objects for fast lookup
    trading_days = set(d.date() for d in market_dates)
    
    # Filter dataframe index to only trading days
    mask = df.index.to_series().apply(lambda x: x.date() in trading_days)
    
    filtered = df[mask].copy()
    
    removed = len(df) - len(filtered)
    print(f"\n{'=' * 60}")
    print("FILTERING TO TRADING DAYS")
    print(f"{'=' * 60}")
    print(f"Before: {len(df):,} rows (all calendar days)")
    print(f"After:  {len(filtered):,} rows (NYSE trading days only)")
    print(f"Removed: {removed:,} rows (weekends + holidays)")
    
    return filtered
# ============================================================
# STEP 3: SUMMARY STATISTICS
# ============================================================

def generate_summary(df: pd.DataFrame) -> pd.DataFrame:
    """Generate comprehensive summary statistics."""
    
    print(f"\n{'=' * 60}")
    print("SUMMARY STATISTICS")
    print(f"{'=' * 60}")
    
    rows = []
    
    for col in df.columns:
        series = df[col]
        n_total = len(series)
        n_valid = series.count()
        n_missing = n_total - n_valid
        pct_missing = (n_missing / n_total) * 100
        pct_coverage = 100 - pct_missing
        
        stats = {
            "column": col,
            "total_rows": n_total,
            "valid": n_valid,
            "missing": n_missing,
            "pct_missing": round(pct_missing, 2),
            "pct_coverage": round(pct_coverage, 2),
        }
        
        if n_valid > 0:
            stats.update({
                "mean": round(series.mean(), 4),
                "std": round(series.std(), 4),
                "min": round(series.min(), 4),
                "p25": round(series.quantile(0.25), 4),
                "median": round(series.median(), 4),
                "p75": round(series.quantile(0.75), 4),
                "max": round(series.max(), 4),
            })
        else:
            stats.update({
                "mean": None, "std": None, "min": None,
                "p25": None, "median": None, "p75": None, "max": None,
            })
        
        rows.append(stats)
    
    summary = pd.DataFrame(rows)
    
    # Determine quality tier
    def quality_tier(pct: float) -> str:
        if pct >= 95:
            return "⭐⭐⭐ EXCELLENT"
        elif pct >= 80:
            return "⭐⭐ GOOD"
        elif pct >= 50:
            return "⭐ FAIR"
        elif pct >= 20:
            return "⚠️ POOR"
        else:
            return "❌ BAD"
    
    summary["quality"] = summary["pct_coverage"].apply(quality_tier)
    
    # Sort by coverage descending
    summary = summary.sort_values("pct_coverage", ascending=False)
    
    # Print overview
    tiers = summary["quality"].value_counts()
    print(f"\nQuality Distribution:")
    for tier, count in tiers.items():
        print(f"  {tier}: {count} columns")
    
    print(f"\nTop 10 (best coverage):")
    print(summary[["column", "pct_coverage", "mean", "std", "quality"]].head(10).to_string(index=False))
    
    print(f"\nBottom 10 (worst coverage):")
    print(summary[["column", "pct_coverage", "mean", "std", "quality"]].tail(10).to_string(index=False))
    
    return summary


# ============================================================
# STEP 4: DROP BAD COLUMNS
# ============================================================

def drop_bad_columns(df: pd.DataFrame, summary: pd.DataFrame, 
                     min_coverage: float = 50.0) -> pd.DataFrame:
    """Drop columns with coverage below threshold."""
    
    bad_cols = summary[summary["pct_coverage"] < min_coverage]["column"].tolist()
    
    print(f"\n{'=' * 60}")
    print(f"DROPPING COLUMNS (< {min_coverage}% COVERAGE)")
    print(f"{'=' * 60}")
    
    if bad_cols:
        print(f"Dropping {len(bad_cols)} columns:")
        for col in bad_cols:
            cov = summary[summary["column"] == col]["pct_coverage"].values[0]
            print(f"  {col}: {cov:.1f}% coverage")
    else:
        print("No columns to drop!")
    
    cleaned = df.drop(columns=bad_cols, errors="ignore")
    print(f"\nBefore: {df.shape[1]} columns")
    print(f"After:  {cleaned.shape[1]} columns")
    
    return cleaned


# ============================================================
# STEP 5: HANDLE REMAINING MISSING VALUES
# ============================================================

def handle_missing(df: pd.DataFrame) -> pd.DataFrame:
    """Forward-fill remaining gaps. NEVER drop trading days."""
    
    print(f"\n{'=' * 60}")
    print("HANDLING REMAINING MISSING VALUES")
    print(f"{'=' * 60}")
    
    before_nan = df.isna().sum().sum()
    print(f"Missing values before: {before_nan:,}")
    print(f"Total trading days: {len(df):,}")
    
    # Forward-fill with generous limit (1 month = ~21 trading days)
    # This fills gaps where data arrives later
    df = df.ffill(limit=21)
    
    after_ffill = df.isna().sum().sum()
    print(f"Missing after ffill (21-day limit): {after_ffill:,}")
    
    # For data that still has leading NaN (series that start after 2000),
    # backfill with a small limit (1 week = 5 trading days)
    # This handles the very beginning of some series
    df = df.bfill(limit=5)
    
    after_bfill = df.isna().sum().sum()
    print(f"Missing after bfill (5-day limit): {after_bfill:,}")
    
    # Only drop rows that are COMPLETELY empty (all NaN)
    # This preserves ALL trading days
    df = df.dropna(how='all')
    
    # For remaining NaN in individual columns, fill with column median
    # (only affects a few edge cases)
    remaining_nan = df.isna().sum().sum()
    if remaining_nan > 0:
        print(f"Filling {remaining_nan:,} remaining NaN with column medians...")
        df = df.fillna(df.median())
    
    print(f"Final shape: {df.shape}")
    print(f"Final date range: {df.index[0]} → {df.index[-1]}")
    print(f"Trading days preserved: {len(df):,} / 6,288")
    
    return df

# ============================================================
# STEP 6: ENGINEER ADDITIONAL FEATURES
# ============================================================

def engineer_additional_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add derived features that benefit from cleaned data."""
    feats = pd.DataFrame(index=df.index)
    
    # Yield Spreads (if not already present)
    if all(c in df.columns for c in ["DGS10", "DGS2"]):
        if "yield_spread_10y2y" not in df.columns:
            feats["yield_spread_10y2y"] = df["DGS10"] - df["DGS2"]
    if all(c in df.columns for c in ["DGS10", "DGS3MO"]):
        if "yield_spread_10y3m" not in df.columns:
            feats["yield_spread_10y3m"] = df["DGS10"] - df["DGS3MO"]
    if all(c in df.columns for c in ["BAA10Y", "AAA10Y"]):
        if "credit_spread_baa_aaa" not in df.columns:
            feats["credit_spread_baa_aaa"] = df["BAA10Y"] - df["AAA10Y"]
    
    # VIX Z-Scores
    if "VIXCLS" in df.columns:
        for w in [20, 60, 252]:
            if f"vix_z_{w}d" not in df.columns:
                roll_mean = df["VIXCLS"].rolling(w).mean()
                roll_std = df["VIXCLS"].rolling(w).std().replace(0, np.nan)
                feats[f"vix_z_{w}d"] = (df["VIXCLS"] - roll_mean) / roll_std
    
    # Regime Flags
    if "yield_spread_10y2y" in df.columns or "yield_spread_10y2y" in feats.columns:
        spread_col = "yield_spread_10y2y"
        spread_data = feats[spread_col] if spread_col in feats.columns else df[spread_col]
        if "regime_yield_inverted" not in df.columns:
            feats["regime_yield_inverted"] = (spread_data < 0).astype(int)
    
    if "VIXCLS" in df.columns:
        if "regime_high_vix" not in df.columns:
            feats["regime_high_vix"] = (df["VIXCLS"] > 25).astype(int)
        if "regime_extreme_vix" not in df.columns:
            feats["regime_extreme_vix"] = (df["VIXCLS"] > 35).astype(int)
    
    if "USRECD" in df.columns:
        if "regime_recession" not in df.columns:
            feats["regime_recession"] = df["USRECD"].fillna(0).astype(int)
    
    if not feats.empty:
        print(f"\n[INFO]: Added {len(feats.columns)} additional derived features")
        df = pd.concat([df, feats], axis=1)
    
    return df


# ============================================================
# MAIN
# ============================================================

def main():
    overall_start = now_ts()
    
    print("=" * 60)
    print("FRED DATA CLEANING PIPELINE")
    print("=" * 60)
    
    # ── Step 1: Load ──
    print("\n[Step 1/6] Loading data...")
    df, market_dates = load_data()
    
    # ── Step 2: Filter to trading days ──
    print("\n[Step 2/6] Filtering to NYSE trading days...")
    df_trading = filter_trading_days(df, market_dates)
    
    # ── Step 3: Summary statistics ──
    print("\n[Step 3/6] Generating summary statistics...")
    summary = generate_summary(df_trading)
    summary.to_csv(OUTPUT_DIR / "column_summary.csv", index=False)
    print(f"\nSummary saved: {OUTPUT_DIR / 'column_summary.csv'}")
    
    # ── Step 4: Drop bad columns ──
    print("\n[Step 4/6] Dropping low-coverage columns...")
    df_clean = drop_bad_columns(df_trading, summary, min_coverage=80.0)    
    
    # ── Step 5: Handle missing ──
    print("\n[Step 5/6] Handling remaining missing values...")
    df_final = handle_missing(df_clean)
    
    # ── Step 6: Additional features ──
    print("\n[Step 6/6] Engineering additional features...")
    df_final = engineer_additional_features(df_final)
    
    # ── Save ──
    output_path = OUTPUT_DIR / "macro_features_trading_days_clean.csv"
    df_final.to_csv(output_path)
    
    # ── Summary ──
    total_elapsed = now_ts() - overall_start
    
    print(f"\n{'=' * 60}")
    print(f"CLEANING PIPELINE COMPLETE — {fmt_elapsed(total_elapsed)}")
    print(f"{'=' * 60}")
    print(f"Output: {output_path}")
    print(f"Shape: {df_final.shape}")
    print(f"Date range: {df_final.index[0].strftime('%Y-%m-%d')} → {df_final.index[-1].strftime('%Y-%m-%d')}")
    print(f"Trading days: {len(df_final):,}")
    print(f"Features: {df_final.shape[1]}")
    
    # Save metadata
    metadata = {
        "date_range": [str(df_final.index[0]), str(df_final.index[-1])],
        "n_trading_days": len(df_final),
        "n_features": df_final.shape[1],
        "feature_names": list(df_final.columns),
        "dropped_columns": list(set(df_trading.columns) - set(df_final.columns)),
    }
    with open(OUTPUT_DIR / "cleaning_metadata.json", "w") as f:
        json.dump(metadata, f, indent=2, default=str)
    print(f"Metadata: {OUTPUT_DIR / 'cleaning_metadata.json'}")
    
    # Final sample
    print(f"\nFinal sample (first 5 rows, first 10 columns):")
    print(df_final.iloc[:5, :10])


if __name__ == "__main__":
    main()